# Linajea × Biohub: train smoke test

Attach the Biohub competition data, enable Internet and a Kaggle GPU, then run all cells. This trains for two iterations on one annotated sample and verifies the saved checkpoint. Output is written only to `/kaggle/working`.

In [ ]:
from pathlib import Path
import subprocess
import sys

DATA_DIR = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development/train')
REPO_DIR = Path('/kaggle/working/linajea')
OUTPUT_DIR = Path('/kaggle/working/biohub_train_smoke')
BRANCH = 'biohub-io'
COMMIT = None  # Set to a pushed commit hash for a reproducible run
SAMPLE = '6bba_05b6850b'
ITERATIONS = 2

if not DATA_DIR.is_dir():
    raise FileNotFoundError(f'Missing data directory: {DATA_DIR}')
for suffix in ('zarr', 'geff'):
    if not (DATA_DIR / f'{SAMPLE}.{suffix}').is_dir():
        raise FileNotFoundError(DATA_DIR / f'{SAMPLE}.{suffix}')
print('Sample:', SAMPLE, 'Output:', OUTPUT_DIR)

In [ ]:
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch',
                    'https://github.com/hathanhv/linajea.git', str(REPO_DIR)], check=True)
if COMMIT is not None:
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', COMMIT], check=True)
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
print('Training code commit:', commit)

In [ ]:
def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', *map(str, args)], check=True)

pip('-r', REPO_DIR / 'requirements-biohub-io.txt')
pip('-r', REPO_DIR / 'requirements-biohub-train.txt')
pip('--no-deps', 'git+https://github.com/abred/daisy@ab7b82054126a50ab4ad5dcf2ef885b02d33aab8')
pip('--no-deps', 'git+https://github.com/Kainmueller-Lab/funlib.learn.torch@36ef666d16bddda0bf3146d4c0efbe9b95c8463d')
pip('-e', REPO_DIR, '--no-deps')

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU before training')
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'run_scripts.train_biohub_smoke',
                '--data-dir', str(DATA_DIR), '--sample', SAMPLE,
                '--output-dir', str(OUTPUT_DIR),
                '--iterations', str(ITERATIONS)],
               cwd=REPO_DIR, check=True)
checkpoint = OUTPUT_DIR / f'train_net_checkpoint_{ITERATIONS}'
print('Checkpoint:', checkpoint, 'bytes:', checkpoint.stat().st_size)
print('Resolved config:', OUTPUT_DIR / 'config.toml')